In [1]:
# ¿Qué evidencia de preferencias contiene FilmTrust y qué tan dispersa está?

import numpy as np
import pandas as pd

raw_ratings = pd.read_csv("../data/filmtrust_ratings.csv")
ratings = raw_ratings.groupby(["user", "movie"], as_index=False)["score"].mean()
user_item = ratings.pivot(index="user", columns="movie", values="score")
density = len(ratings) / (user_item.shape[0] * user_item.shape[1])
pd.DataFrame({"raw_ratings": [len(raw_ratings)], "consolidated_ratings": [len(ratings)], "users": [user_item.shape[0]], "movies": [user_item.shape[1]], "matrix_density": [density]})

,raw_ratings,consolidated_ratings,users,movies,matrix_density
0,35497,35494,1508,2071,0.011365


In [2]:
# ¿Qué usuarios aportan suficiente historial para encontrar preferencias comparables?

user_activity = ratings.groupby("user").agg(
    ratings=("score", "size"),
    mean_score=("score", "mean"),
).sort_values("ratings", ascending=False)
user_activity.head(10)

,ratings,mean_score
user,,
272,244,2.694672
1187,232,2.745690
161,197,2.743655
3,193,2.829016
969,174,2.784483
79,172,2.409884
199,155,2.061290
702,147,3.125850
670,137,3.072993


In [3]:
# ¿Cómo se comparan preferencias sin confundir una ausencia de calificación con una baja calificación?

def cosine_similarity(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    normalized = np.divide(matrix, norms, out=np.zeros_like(matrix, dtype=float), where=norms != 0)
    return normalized @ normalized.T

target_user = 272
minimum_common_ratings = 2
minimum_similarity = 0.05
observed = user_item.notna().astype(int)
user_means = user_item.mean(axis=1)
centered_ratings = user_item.sub(user_means, axis=0).fillna(0)

In [4]:
# ¿Qué vecinos de la persona 272 tienen al menos dos películas en común y afinidad positiva?

similarity = pd.DataFrame(
    cosine_similarity(centered_ratings.to_numpy()),
    index=user_item.index,
    columns=user_item.index,
)
common_ratings = pd.DataFrame(
    observed.to_numpy() @ observed.to_numpy().T,
    index=user_item.index,
    columns=user_item.index,
)
neighbors = pd.DataFrame({
    "neighbor_user": similarity.index,
    "similarity": similarity.loc[target_user].to_numpy(),
    "common_ratings": common_ratings.loc[target_user].to_numpy(),
    "neighbor_ratings": observed.sum(axis=1).to_numpy(),
}).query("neighbor_user != @target_user")
eligible_neighbors = neighbors.query(
    "common_ratings >= @minimum_common_ratings and similarity >= @minimum_similarity"
).sort_values("similarity", ascending=False)
eligible_neighbors.head(10)

,neighbor_user,similarity,common_ratings,neighbor_ratings
1179,1180,0.200902,10,19
1146,1147,0.191833,59,86
314,315,0.188271,6,16
512,513,0.179502,4,5
211,212,0.174237,11,19
19,20,0.170599,3,3
65,66,0.168830,5,8
268,269,0.166855,6,7
141,142,0.163595,9,16
706,707,0.163558,3,6


In [5]:
# ¿Qué películas no valoradas tienen una estimación respaldada por al menos dos vecinos?

weighted_deviations = pd.Series(0.0, index=user_item.columns)
weight_sums = pd.Series(0.0, index=user_item.columns)
support_counts = pd.Series(0, index=user_item.columns, dtype=int)
for neighbor in eligible_neighbors.itertuples(index=False):
    deviations = user_item.loc[neighbor.neighbor_user] - user_means.loc[neighbor.neighbor_user]
    available = deviations.notna()
    weighted_deviations.loc[available] += neighbor.similarity * deviations.loc[available]
    weight_sums.loc[available] += neighbor.similarity
    support_counts.loc[available] += 1

predictions = user_means.loc[target_user] + weighted_deviations.divide(weight_sums.where(weight_sums != 0))
recommendations = pd.DataFrame({
    "movie_id": predictions.index,
    "predicted_score": predictions.values,
    "supporting_neighbors": support_counts.values,
})
recommendations = recommendations.loc[
    user_item.loc[target_user].isna().to_numpy()
    & recommendations["predicted_score"].notna()
    & recommendations["supporting_neighbors"].ge(2)
] .sort_values(["predicted_score", "supporting_neighbors"], ascending=False)
recommendations.head(10)

,movie_id,predicted_score,supporting_neighbors
869,870,3.896019,2
308,309,3.876018,2
1188,1189,3.806561,2
808,809,3.694891,2
921,922,3.690060,2
398,399,3.689661,2
1181,1182,3.633825,2
833,834,3.627928,2
1037,1038,3.587162,3
688,689,3.585083,4


In [6]:
# Se conservan la cobertura, los vecinos seleccionados y las recomendaciones como evidencia del taller.

from pathlib import Path

submission_dir = Path("../submission")
coverage = pd.DataFrame([
    {"metric": "raw_ratings", "value": len(raw_ratings)},
    {"metric": "consolidated_ratings", "value": len(ratings)},
    {"metric": "users", "value": user_item.shape[0]},
    {"metric": "movies", "value": user_item.shape[1]},
    {"metric": "matrix_density", "value": density},
    {"metric": "target_user", "value": target_user},
    {"metric": "target_ratings", "value": user_item.loc[target_user].notna().sum()},
    {"metric": "eligible_neighbors", "value": len(eligible_neighbors)},
])
coverage.to_csv(submission_dir / "coverage_summary.csv", index=False)
eligible_neighbors.to_csv(submission_dir / "nearest_neighbors.csv", index=False)
recommendations.to_csv(submission_dir / "recommendations.csv", index=False)

In [7]:
# ¿La predicción colaborativa aporta frente a un promedio simple por película en calificaciones que el método no vio?

rng_holdout = np.random.default_rng(0)
holdout_fraction = 0.15
minimum_ratings_for_holdout = 10

eligible_for_holdout = user_activity.query("ratings >= @minimum_ratings_for_holdout").index
holdout_pool = ratings[ratings["user"].isin(eligible_for_holdout)]

holdout_positions = rng_holdout.choice(
    holdout_pool.index.to_numpy(),
    size=int(len(holdout_pool) * holdout_fraction),
    replace=False,
)
holdout_ratings = ratings.loc[holdout_positions].reset_index(drop=True)
training_ratings = ratings.drop(index=holdout_positions).reset_index(drop=True)

len(training_ratings), len(holdout_ratings)

(30489, 5005)

In [8]:
# Las medias, similitudes y vecinos se recalculan igual que antes, pero sólo con las calificaciones de entrenamiento.

user_item_train = training_ratings.pivot(index="user", columns="movie", values="score")
observed_train = user_item_train.notna().astype(int)
user_means_train = user_item_train.mean(axis=1)
centered_train = user_item_train.sub(user_means_train, axis=0).fillna(0)
similarity_train = pd.DataFrame(
    cosine_similarity(centered_train.to_numpy()),
    index=user_item_train.index,
    columns=user_item_train.index,
)
common_ratings_train = pd.DataFrame(
    observed_train.to_numpy() @ observed_train.to_numpy().T,
    index=user_item_train.index,
    columns=user_item_train.index,
)
movie_means_train = user_item_train.mean(axis=0)
global_mean_train = training_ratings["score"].mean()

In [9]:
# La misma regla de vecinos y puntuación de H02–H03 se aplica a cada usuario con una calificación retenida, no sólo a la persona 272.

minimum_supporting_neighbors = 2


def predict_for_user(user_id):
    neighbors = pd.DataFrame({
        "neighbor_user": similarity_train.index,
        "similarity": similarity_train.loc[user_id].to_numpy(),
        "common_ratings": common_ratings_train.loc[user_id].to_numpy(),
    }).query("neighbor_user != @user_id")
    eligible = neighbors.query(
        "common_ratings >= @minimum_common_ratings and similarity >= @minimum_similarity"
    )

    weighted_deviations = pd.Series(0.0, index=user_item_train.columns)
    weight_sums = pd.Series(0.0, index=user_item_train.columns)
    support_counts = pd.Series(0, index=user_item_train.columns, dtype=int)
    for neighbor in eligible.itertuples(index=False):
        deviations = user_item_train.loc[neighbor.neighbor_user] - user_means_train.loc[neighbor.neighbor_user]
        available = deviations.notna()
        weighted_deviations.loc[available] += neighbor.similarity * deviations.loc[available]
        weight_sums.loc[available] += neighbor.similarity
        support_counts.loc[available] += 1

    predicted = user_means_train.loc[user_id] + weighted_deviations.divide(weight_sums.where(weight_sums != 0))
    return predicted, support_counts


collaborative_predictions = {
    user_id: predict_for_user(user_id)
    for user_id in holdout_ratings["user"].unique()
    if user_id in similarity_train.index
}
len(collaborative_predictions)

949

In [10]:
# Se predice cada calificación retenida con la línea base de película y con el método colaborativo, con las mismas reglas de respaldo.

def baseline_prediction(movie_id):
    if movie_id in movie_means_train.index and pd.notna(movie_means_train.loc[movie_id]):
        return movie_means_train.loc[movie_id]
    return global_mean_train


def collaborative_prediction(user_id, movie_id):
    predicted, support_counts = collaborative_predictions.get(user_id, (None, None))
    if predicted is None or movie_id not in predicted.index:
        return np.nan
    if support_counts.loc[movie_id] < minimum_supporting_neighbors:
        return np.nan
    return predicted.loc[movie_id]


holdout_ratings["baseline_prediction"] = holdout_ratings["movie"].map(baseline_prediction)
holdout_ratings["collaborative_prediction"] = holdout_ratings.apply(
    lambda row: collaborative_prediction(row["user"], row["movie"]), axis=1
)
holdout_ratings.head()

,user,movie,score,baseline_prediction,collaborative_prediction
0,471,13,4.0,3.250716,3.148431
1,174,7,3.5,3.156566,1.982282
2,900,8,2.0,3.211604,2.173076
3,1318,11,4.0,3.401235,3.992723
4,1157,218,2.0,3.570312,3.473417


In [11]:
# La cobertura de cada método se mide sobre todas las retenidas; el error sólo se compara donde ambos métodos predicen, para que la comparación sea pareja.

from sklearn.metrics import mean_absolute_error, mean_squared_error

holdout_ratings["baseline_covered"] = holdout_ratings["baseline_prediction"].notna()
holdout_ratings["collaborative_covered"] = holdout_ratings["collaborative_prediction"].notna()
comparable = holdout_ratings[
    holdout_ratings["baseline_covered"] & holdout_ratings["collaborative_covered"]
]

holdout_evaluation = pd.DataFrame([
    {
        "method": "Línea base (promedio de película)",
        "n_predicted": int(holdout_ratings["baseline_covered"].sum()),
        "coverage": holdout_ratings["baseline_covered"].mean(),
        "mae": mean_absolute_error(comparable["score"], comparable["baseline_prediction"]),
        "rmse": mean_squared_error(comparable["score"], comparable["baseline_prediction"], squared=False),
    },
    {
        "method": "Filtrado colaborativo",
        "n_predicted": int(holdout_ratings["collaborative_covered"].sum()),
        "coverage": holdout_ratings["collaborative_covered"].mean(),
        "mae": mean_absolute_error(comparable["score"], comparable["collaborative_prediction"]),
        "rmse": mean_squared_error(comparable["score"], comparable["collaborative_prediction"], squared=False),
    },
]).round(3)

holdout_evaluation

/Volumes/GitHub/classroom-vibecoding/.venv/lib/python3.9/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/Volumes/GitHub/classroom-vibecoding/.venv/lib/python3.9/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


,method,n_predicted,coverage,mae,rmse
0,Línea base (promedio de película),5005,1.000,0.694,0.881
1,Filtrado colaborativo,4308,0.861,0.591,0.775


In [12]:
# Se conserva la comparación retenida como evidencia de si el filtrado colaborativo aporta frente a un promedio simple.

holdout_evaluation.to_csv(submission_dir / "holdout_evaluation.csv", index=False)

# Sobre 5,005 calificaciones retenidas, el filtrado colaborativo predice con menor error que
# el promedio de película en las retenidas que ambos métodos pueden predecir (MAE 0.591 frente
# a 0.694; RMSE 0.775 frente a 0.881), con una cobertura de 86.1 % frente al 100 % trivial de
# la línea base, que siempre tiene un respaldo (la media global). La comparación muestra que la
# afinidad entre usuarios aporta sobre un promedio simple en este conjunto retenido; no mide
# satisfacción, calidad del orden de las recomendaciones ni impacto en una decisión real de un
# usuario.